# Web News Pipeline Research Replica

This notebook runs the Web News pipeline in a research-safe way:

1. search Google News RSS for a small keyword set
2. resolve article URLs and extract bodies using `web_agent.py`
3. write raw results under `research/output/`
4. optionally run the same `engine.py --web` style generation
5. score bilinguality and Hindi output quality

Set `RUN_WEB_SEARCH = True` and `RUN_AI = True` only when you want live web/API calls.

In [ ]:
from pathlib import Path
import json
import os
import re
import sys
from collections import Counter

# Locate the research package by walking up, so this notebook works whether the
# kernel's working directory is research/notebooks/, research/ or the repo root.
def _find_research(start):
    for p in (start, *start.parents):
        for cand in (p, p / 'research'):
            if (cand / 'shared').is_dir() and (cand / 'web_search').is_dir():
                return cand
    raise RuntimeError('cannot locate the research package from %s' % start)

RESEARCH_DIR = _find_research(Path.cwd().resolve())
PROJECT_ROOT = RESEARCH_DIR.parent
# Both are needed: PROJECT_ROOT for the webapp modules (engine, web_agent),
# RESEARCH_DIR for the research packages (web_search, rsb_search, shared).
for _p in (PROJECT_ROOT, RESEARCH_DIR):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

OUTPUT_DIR = RESEARCH_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CONFIG_FILE = RESEARCH_DIR / 'hindi_railway_sources.json'
WEB_RAW_FILE = OUTPUT_DIR / 'web_crawl_output_research.json'
WEB_AI_FILE = OUTPUT_DIR / 'web_agency_output_research.json'
WEB_SEEN_FILE = OUTPUT_DIR / 'web_seen_articles_research.json'
WEB_STATUS_FILE = OUTPUT_DIR / 'web_pipeline_status_research.json'

RUN_WEB_SEARCH = False   # Google News RSS path (web_agent) - currently cannot resolve URLs
RUN_DIRECT_CRAWL = False # direct publisher tag-page crawl (research/web_search/pipeline)
RUN_AI = False

source_config = json.loads(SOURCE_CONFIG_FILE.read_text(encoding='utf-8'))
WEB_SEARCH_SOURCES = source_config['source_groups']['web_search_sources']['sources']
WEB_SEARCH_KEYWORDS = []
for source in WEB_SEARCH_SOURCES:
    WEB_SEARCH_KEYWORDS.extend(source.get('search_keywords', []))
WEB_SEARCH_KEYWORDS = list(dict.fromkeys(WEB_SEARCH_KEYWORDS))

RESEARCH_SETTINGS = {
    'keywords': WEB_SEARCH_KEYWORDS[:4],  # small default sample; increase for broader research
    'max_articles': 5,
    'per_keyword': 3,
    'date_from': '',  # web_agent format: yyyy-mm-dd
    'date_to': '',    # web_agent format: yyyy-mm-dd
}

print('Project root:', PROJECT_ROOT)
print('Research output:', OUTPUT_DIR)
print('Web source config:', SOURCE_CONFIG_FILE)
print('Available web sources:', [s['name'] for s in WEB_SEARCH_SOURCES])
print('Selected web keywords:', RESEARCH_SETTINGS['keywords'])


def load_json_file(path):
    """Read a research JSON file, tolerating a missing / empty / truncated file.

    engine.run_generate can leave a 0-byte output behind if a run is interrupted,
    and json.loads('') raises JSONDecodeError.
    """
    from pathlib import Path as _P
    p = _P(path)
    if not p.exists() or p.stat().st_size == 0:
        return None
    try:
        return json.loads(p.read_text(encoding='utf-8'))
    except json.JSONDecodeError as e:
        print('WARNING: %s is not valid JSON (%s). Ignoring it.' % (p.name, e))
        return None


## Stage 1: Article Discovery And Body Extraction

Two routes:

- `RUN_WEB_SEARCH` uses the webapp's `web_agent.run_web_search()` (Google News RSS +
  Brave). Google's encrypted redirects currently resolve for none of the six Hindi
  sources, so bodies fall back to the ~150-char RSS snippet.

- `RUN_DIRECT_CRAWL` uses `research/web_search/pipeline.py`, which skips Google entirely and
  reads each publisher's own tag/topic page. Verified on all six sources.

Both write only into `research/output/`. The webapp is never modified.


In [ ]:
import web_agent
from web_search import pipeline as hp
from shared.hindi_patches import apply_hindi_patches

# The webapp's normalizers delete Devanagari (dedup._normalise, _make_dedup_key) and
# tokenize Latin-only (_title_words). Rebind them before any Hindi runs through.
apply_hindi_patches()

# Redirect web-agent output away from app production files.
web_agent.WEB_CRAWL_OUTPUT = str(WEB_RAW_FILE)
web_agent.WEB_DEDUP_FILE = str(WEB_SEEN_FILE)
web_agent.STATUS_FILE = str(WEB_STATUS_FILE)
web_agent.load_web_settings = lambda: RESEARCH_SETTINGS

# Research runs should not inherit the webapp's persistent seen-article history,
# otherwise a second run silently returns fewer articles than the first.
web_agent._load_web_seen = lambda: set()
web_agent._save_web_seen = lambda seen: None

if RUN_DIRECT_CRAWL:
    raw_articles = hp.run_hindi_crawl(max_per_source=3, out_file=str(WEB_RAW_FILE))
elif RUN_WEB_SEARCH:
    raw_articles = web_agent.run_web_search()
else:
    raw_articles = load_json_file(WEB_RAW_FILE)
    if raw_articles is None:
        raw_articles = []
        print('No research crawl yet. Set RUN_DIRECT_CRAWL = True to fetch.')
        print('NOTE: the app files in the project root are NOT loaded as a fallback --\n'
              '      they hold English keyword results and would distort Hindi scores.')
    else:
        print('Loaded previous research crawl:', WEB_RAW_FILE.name)

print('Raw web articles:', len(raw_articles))
if raw_articles:
    lens = sorted(len(a.get('body', '')) for a in raw_articles)
    print('Body chars: min=%d median=%d max=%d' % (lens[0], lens[len(lens)//2], lens[-1]))


### Hindi preprocessing check

Proves the patches are active and reach the webapp's internal callers. `similar_count == 2`
is the meaningful assertion — it shows `_group_similar_articles` picked up the patched
tokenizer, which is the whole point of the monkeypatch.


In [ ]:
from shared.hindi_patches import verify_patches

_m = verify_patches()

assert _m['keys_after'] == _m['titles'], 'dedup keys still colliding'
assert _m['empty_wordsets_after'] == 0, 'titles still untokenizable'
assert _m['grouping_similar_count'] == 2, 'patch did not reach _group_similar_articles'
assert _m['nukta_variants_hash_equal'], 'nukta encodings still hash differently'
print('\nAll Hindi preprocessing assertions passed.')


## Stage 2: AI Generation

This calls the same `engine.run_generate()` used by `POST /web-generate`, but reads/writes research files.

In [ ]:
import engine
from shared import config_env

engine.STATUS_FILE = str(WEB_STATUS_FILE)

if RUN_AI:
    # Key resolution lives in research/config_env.py: shell export > research/.env >
    # project-root .env (loaded by config.py). Never hardcode a key here -- this
    # notebook IS tracked by git and would carry the secret into the repo.
    if not config_env.apply_to_engine(engine):
        print('Generation would return empty results.')
    # engine skips titles already present in the destination file; delete WEB_AI_FILE
    # to force a clean regeneration after a prompt change.
    ai_articles = engine.run_generate(str(WEB_RAW_FILE), str(WEB_AI_FILE))
else:
    ai_articles = load_json_file(WEB_AI_FILE)
    if ai_articles is None:
        ai_articles = raw_articles
        print('No research AI output yet; scoring raw crawl instead.')

print('Research AI articles:', len(ai_articles))
print('AI file:', WEB_AI_FILE)


## Stage 3: Hindi And Bilingual Scores

In [ ]:
devanagari_re = re.compile(r'[\u0900-\u097F]')
english_re = re.compile(r'[A-Za-z]')

rows = []
for idx, art in enumerate(ai_articles):
    source_lang = art.get('source_lang', 'unknown')
    title = art.get('title', '') or ''
    body = art.get('body', '') or ''
    story = art.get('ai_news_story', '') or ''
    translated = art.get('ai_news_story_translated', '') or ''

    story_has_hindi = bool(devanagari_re.search(story))
    translation_has_hindi = bool(devanagari_re.search(translated))
    story_has_english = bool(english_re.search(story))
    translation_has_english = bool(english_re.search(translated))
    source_has_hindi = bool(devanagari_re.search(title + ' ' + body))

    direction_ok = False
    if source_lang == 'hi':
        direction_ok = story_has_hindi and translation_has_english
    elif source_lang == 'en':
        direction_ok = story_has_english and translation_has_hindi

    bilingual_score = 0
    bilingual_score += 25 if source_lang in ('hi', 'en') else 0
    bilingual_score += 25 if len(story.strip()) > 30 else 0
    bilingual_score += 25 if len(translated.strip()) > 30 else 0
    bilingual_score += 25 if direction_ok else 0

    hindi_information_score = 0
    hindi_information_score += 35 if source_has_hindi else 0
    hindi_information_score += 35 if story_has_hindi else 0
    hindi_information_score += 30 if translation_has_hindi else 0

    rows.append({
        'idx': idx,
        'source': art.get('source', ''),
        'date': art.get('date', ''),
        'source_lang': source_lang,
        'bilingual_score': bilingual_score,
        'hindi_information_score': hindi_information_score,
        'fact_score': art.get('fact_score', None),
        'plagiarism_score': art.get('plagiarism_score', None),
        'story_has_hindi': story_has_hindi,
        'translation_has_hindi': translation_has_hindi,
        'validated_sources': len(art.get('validated_sources', []) or []),
        'similar_count': art.get('similar_count', 1),
        'title': title[:120],
    })

def avg(values):
    values = [v for v in values if isinstance(v, (int, float))]
    return round(sum(values) / len(values), 2) if values else None

print('Total:', len(rows))
print('Source language counts:', dict(Counter(r['source_lang'] for r in rows)))
print('Top sources:', Counter(r['source'] for r in rows).most_common(10))
print('Average bilingual score:', avg([r['bilingual_score'] for r in rows]))
print('Average Hindi information score:', avg([r['hindi_information_score'] for r in rows]))
print('Average fact score:', avg([r['fact_score'] for r in rows]))
print('Average plagiarism score:', avg([r['plagiarism_score'] for r in rows]))

for row in rows[:25]:
    print(row)

## Stage 4: Grounded Quality Scores

`fact_score`, `impact_level` and `topics` are all produced by the same Claude call that
writes the story — nothing verifies them against the source. Measured over 25 articles,
`fact_score` has only 5 distinct values with 88% in 92-95.

`research/shared/grounding.py` adds deterministic, source-checked fields alongside them:
`coverage_score` (share of source figures the story carried), a hallucination gate,
topic vocabulary + evidence checks, and a rule-derived impact level.


In [ ]:
from shared.score_report import rescore_file
from shared import grounding

scored, scored_path = rescore_file(str(WEB_AI_FILE))
print('scored ->', scored_path)

print('\n%-4s %-6s %-9s %-6s %-7s %s' % ('#','fact','coverage','prec','impact','gates'))
print('-'*72)
for i, a in enumerate(scored):
    print('%-4d %-6s %-9s %-6s %-7s %s' % (
        i, a.get('fact_score'), a.get('coverage_score'), a.get('number_precision'),
        '%s/%s' % (a.get('impact_level'), a.get('impact_rule_level')),
        'OK' if a.get('gates_passed') else '; '.join(a.get('gate_failures', []))[:36]))

_cov = [a['coverage_score'] for a in scored if a.get('coverage_score') is not None]
_fac = [a['fact_score'] for a in scored if a.get('fact_score') is not None]
print('\ndistinct values -> fact_score: %d | coverage_score: %d'
      % (len(set(_fac)), len(set(_cov))))
